# Mixed-size RL training and checkpoint comparison (v7)

Train three settings × three seeds on small and large configurations. Validate
saved checkpoints, select one setting/episode across all seeds, and compare with
your original v5 policies on fresh cases. The planner models really train;
distributed workloads remain simulated. See `docs/PILOT_V7.md`.

No mandatory time limit. Parallel CPU workers. Separate training and evaluation
cells; full test evaluation is an explicit later choice.


In [ ]:
from pathlib import Path
import os, sys, json
for key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[key] = "1"
ROOT = Path.cwd()
if not (ROOT / "learnedplacement").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "learnedplacement").is_dir()
sys.path.insert(0, str(ROOT))
# Install once: python -m pip install -e '.[notebook,learning]'
import pandas as pd
from learnedplacement.runner import read_json
from learnedplacement import tuning_runner as tuning
from learnedplacement import guided_runner as guided


## Editable training settings

384 training cases cover pools of 6/12/24/32/64/96 machines. Checkpoint validation
uses 24 separate cases, 16 planning traces and 64 evaluation traces each.
Defaults: 9 training jobs, each up to 16,384 episodes × 8 rollouts. This is more
work than v5; completed progress is reusable and CPU jobs run concurrently.

The reward stays unchanged. Checkpoint selection uses mean top-eight validation
objective across all seeds; completion time, cost and screens are reported too.
Do not change settings midway through an existing output folder.


In [ ]:
WORKERS = min(4, os.cpu_count() or 1)
MAX_MINUTES = None  # Unlimited; optional positive time limit is supported
TUNING_RUN_NAME = f"pilot_v7_tuning_w{WORKERS}"
CONFIG = read_json(ROOT / "configs" / "pilot_v7.json")
CONFIG["tuning"]["checkpoints"] = [2048, 4096, 8192, 16384]
CONFIG["learning"]["rollouts_per_episode"] = 8
CONFIG["tuning"]["selection_metric"] = "objective"
CONFIG["tuning"]["variants"] = [
    {"name": "baseline", "hidden": 32, "learning_rate": 0.001, "entropy": 0.01},
    {"name": "wider", "hidden": 64, "learning_rate": 0.001, "entropy": 0.01},
    {"name": "conservative", "hidden": 64, "learning_rate": 0.0003, "entropy": 0.03},
]
tuning.validate_config(CONFIG)
TUNING_OUTPUT = ROOT / "results" / TUNING_RUN_NAME
print("Training output:", TUNING_OUTPUT)
print("Workers:", WORKERS, "Optional time limit:", MAX_MINUTES)
print(json.dumps(CONFIG, indent=2))


## Train / validate checkpoints / resume

This cell starts actual RL training and checkpoint validation. No test cases are
used. Rerun unchanged to resume. A handled interruption saves the current state;
a forced kill can lose up to 31 episodes since the last periodic save. Logs and
states are under `jobs/<setting>-seed-<n>/`. Interrupts may wait for workers.


In [ ]:
result = tuning.run(config=CONFIG, output=TUNING_OUTPUT,
                    workers=WORKERS, max_minutes=MAX_MINUTES)
print(json.dumps(result, indent=2))


## Saved training progress and validation curves

This cell reads saved experiments and refreshes the derived summary; it does not
train. Curves show recent reward (not only cumulative reward), validation
completion time and validation objective. Seed averages require all seeds.


In [ ]:
report = tuning.summary(TUNING_OUTPUT)
print(report["state"], report["validation_checkpoints"], "/", report["expected_validation_checkpoints"], "checkpoint validations")
display(pd.DataFrame([{k: v for k, v in row.items() if k != "history"}
                      for row in report["jobs"]]))
display(tuning.plot_progress(TUNING_OUTPUT))
if (TUNING_OUTPUT / "selection.json").exists():
    selection = read_json(TUNING_OUTPUT / "selection.json")
    print("Selected setting and checkpoint:", selection["selection"])
    display(pd.DataFrame(selection["groups"]))


## Freeze original and selected policies for comparison

Run after tuning finishes. SOURCE_RUN must be your original completed v5 results
folder containing weights, not an analysis ZIP. Auto-discovery requires exactly
one distinct completed v5 model set; otherwise set the path explicitly.

IDs 11/29/47 are the old small-trained policies. IDs 1011/1029/1047 identify the
new family trained with those same seeds. All seeds are kept; none is picked as
the "best seed". The shared setting/episode is selected on tuning validation only.


In [ ]:
SOURCE_RUN = None  # e.g. ROOT / "results" / "pilot_v5_8192"
if SOURCE_RUN is None:
    SOURCE_RUN = guided.find_model_run(ROOT / "results")
COMPARISON_SOURCE = ROOT / "results" / f"pilot_v7_models_w{WORKERS}"
tuning.comparison_source(TUNING_OUTPUT, SOURCE_RUN, COMPARISON_SOURCE)
print("Old policies from:", SOURCE_RUN)
print("Comparison models:", read_json(COMPARISON_SOURCE / "comparison_policies.json")["model_labels"])


## Fresh full comparison settings

Start with validation. This uses a different hardware seed from training/tuning,
all six sizes and all four profiles: 96 cases, 16 methods, 256 evaluation traces.
Both old and new policies see exactly the same action pools and failure traces.
Compare RL top eight with random and analytical top eight, full-pool search and
the longer local-search reference. Keep test untouched until choices are final.


In [ ]:
SPLIT = "validation"  # Later: "test", with a separate output directory
EVAL_CONFIG = tuning.evaluation_config(CONFIG, COMPARISON_SOURCE, split=SPLIT)
EVAL_CONFIG["planning_traces"] = 32
EVAL_CONFIG["evaluation_traces"] = 256
guided.validate_config(EVAL_CONFIG)
OUTPUT = ROOT / "results" / f"pilot_v7_{SPLIT}_w{WORKERS}"
print("Comparison output:", OUTPUT)
print("Methods:", EVAL_CONFIG["methods"])


## Run / resume the fresh comparison


In [ ]:
result = guided.run(config=EVAL_CONFIG, output=OUTPUT,
                    source_run=COMPARISON_SOURCE, workers=WORKERS, max_minutes=MAX_MINUTES)
print(json.dumps(result, indent=2))


## Read saved comparison results and plots

Use each size/profile separately. Only fully matched cases enter plots. Lower
completion time can trade off against cost/deadline penalties; inspect both.
Parallel CPU contention affects measured latency. Training cost is separate.


In [ ]:
print(json.dumps(guided.status(OUTPUT), indent=2))
PROFILE = "reference"  # Or slow_io, slow_network, variable_repair, None
AVAILABLE_NODES = 64   # 6, 12, 24, 32, 64, 96; None gives size curves
if (OUTPUT / "summary.csv").exists():
    results = pd.read_csv(OUTPUT / "summary.csv")
    display(results[["case_id", "method", "restricted_mean_s", "mean_observed_cost_usd",
                     "statistical_screen_passed", "censored_traces", "planning_s"]])
    display(guided.plot_results(OUTPUT, profile=PROFILE, available_nodes=AVAILABLE_NODES))


## Compact exports

Share both ZIPs: training curves/selection and fresh paired evaluation. The first
export also works before evaluation has started. Raw rollouts and weights stay
local; old outputs are preserved.


In [ ]:
training_archive = tuning.export(TUNING_OUTPUT)
print("Training analysis:", training_archive)
if "OUTPUT" in globals() and (OUTPUT / "manifest.json").exists():
    evaluation_archive = guided.export(OUTPUT)
    print("Comparison analysis:", evaluation_archive)
